In [1]:
# ==========================================================
# Wine Quality - FLAML (STREAMLIT READY VERSION)
# Dataset: https://www.kaggle.com/datasets/yasserh/wine-quality-dataset
#
# ✅ AutoML FLAML com tempo controlado (sala de aula)
# ✅ Feature Engineering manual
# ✅ Pipeline completo (preprocess + modelo)
# ✅ Exportação PADRÃO para Streamlit único
# ==========================================================

import os
import json
import glob
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score

from ydata_profiling import ProfileReport
import kagglehub
import joblib

from flaml import AutoML

# =========================
# CONFIG
# =========================
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIME_BUDGET = 120  # segundos (ideal para sala de aula)

BASE_DIR = os.path.abspath("wine_flaml")
DIR_REPORTS = os.path.join(BASE_DIR, "reports")
DIR_FIGURES = os.path.join(BASE_DIR, "figures")

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DIR_REPORTS, exist_ok=True)
os.makedirs(DIR_FIGURES, exist_ok=True)

print("✅ FLAML – versão pronta para Streamlit único")

# =========================
# AUX
# =========================
def load_dataset():
    path = kagglehub.dataset_download("yasserh/wine-quality-dataset")
    csv = [c for c in glob.glob(os.path.join(path, "*.csv")) if "WineQT" in c][0]
    return pd.read_csv(csv).drop(columns=["Id"])


def normalize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


def feature_engineering(df):
    eps = 1e-9
    df = df.copy()
    df["total_acidity"] = df["fixed_acidity"] + df["volatile_acidity"] + df["citric_acid"]
    df["alcohol_sugar_ratio"] = df["alcohol"] / (df["residual_sugar"] + eps)
    df["so2_ratio"] = df["free_sulfur_dioxide"] / (df["total_sulfur_dioxide"] + eps)
    df["density_alcohol"] = df["density"] * df["alcohol"]
    return df


def plot_balance(y):
    plt.figure(figsize=(6, 4))
    sns.countplot(x=y, color="maroon")
    plt.title("Distribuição das Classes (quality)")
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_FIGURES, "class_balance.png"), dpi=200)
    plt.close()


# =========================
# MAIN
# =========================
def main():
    print("[1/5] Carregando dados...")
    df = load_dataset()
    df = normalize_columns(df)

    print("[2/5] EDA...")
    ProfileReport(df, title="Wine Quality - EDA (FLAML)") \
        .to_file(os.path.join(DIR_REPORTS, "eda_wine_quality.html"))

    plot_balance(df["quality"])

    print("[3/5] Feature Engineering...")
    df = feature_engineering(df)

    # Target encoding
    le = LabelEncoder()
    y = le.fit_transform(df["quality"])
    X = df.drop(columns=["quality"])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE
    )

    # Preprocessador
    preprocessor = ColumnTransformer(
        [("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), X.columns)]
    )

    X_train_p = preprocessor.fit_transform(X_train)
    X_test_p = preprocessor.transform(X_test)

    print("[4/5] AutoML FLAML...")
    automl = AutoML()

    automl.fit(
        X_train_p,
        y_train,
        task="classification",
        metric="macro_f1",
        time_budget=TIME_BUDGET,
        seed=RANDOM_STATE,
        estimator_list=[
            "lgbm",
            "xgboost",
            "xgb_limitdepth",
            "rf",
            "extra_tree",
            "histgb",
            "catboost",
            "kneighbor",
            "svc",
            "sgd",
            "lrl2",
            "lrl1",
        ]
    )

    y_pred = automl.predict(X_test_p)
    f1 = f1_score(y_test, y_pred, average="macro")

    print("\n✅ F1-macro:", round(f1, 4))
    print(classification_report(y_test, y_pred))

    # =========================================================
    # PIPELINE FINAL PARA STREAMLIT
    # =========================================================
    final_pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", automl.model)
    ])

    artifact_name = "model_streamlit.joblib"
    joblib.dump(final_pipeline, os.path.join(BASE_DIR, artifact_name))

    # =========================================================
    # META.JSON PADRONIZADO (STREAMLIT ÚNICO)
    # =========================================================
    meta = {
        "model_kind": "sklearn",               # FLAML → sklearn-like
        "artifact_path": artifact_name,
        "target": "quality",
        "features": X.columns.tolist(),
        "classes": list(range(len(le.classes_))),
        "original_classes": le.classes_.tolist(),

        # extras didáticos
        "framework": "flaml",
        "best_model": automl.model.__class__.__name__,
        "macro_f1": f1,
        "time_budget_seconds": TIME_BUDGET
    }

    with open(os.path.join(BASE_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print("\n✅ FLAML FINALIZADO COM SUCESSO")
    print("Estrutura gerada em wine_flaml/:")
    print("- model_streamlit.joblib")
    print("- meta.json")
    print("- reports/eda_wine_quality.html")
    print("- figures/class_balance.png")
    print("\n➡ Próximo passo: streamlit run app_streamlit.py")


# =========================
if __name__ == "__main__":
    main()

✅ FLAML – versão pronta para Streamlit único
[1/5] Carregando dados...
[2/5] EDA...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:00<?, ?it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

[3/5] Feature Engineering...
[4/5] AutoML FLAML...
[flaml.automl.logger: 05-03 23:14:43] {2375} INFO - task = classification
[flaml.automl.logger: 05-03 23:14:43] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 05-03 23:14:43] {2489} INFO - Minimizing error metric: 1-macro_f1
[flaml.automl.logger: 05-03 23:14:43] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'xgboost', 'xgb_limitdepth', 'rf', 'extra_tree', 'histgb', 'catboost', 'kneighbor', 'svc', 'sgd', 'lrl2', 'lrl1']
[flaml.automl.logger: 05-03 23:14:43] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 05-03 23:14:43] {3046} INFO - Estimated sufficient time budget=3065s. Estimated necessary time budget=93s.
[flaml.automl.logger: 05-03 23:14:43] {3097} INFO -  at 0.3s,	estimator lgbm's best error=7.9332e-01,	best estimator lgbm's best error=7.9332e-01
[flaml.automl.logger: 05-03 23:14:43] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 05-03 23:14:43] {3097} INFO -  at 

  File "C:\Users\calab\.venv310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\calab\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\calab\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\calab\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


[flaml.automl.logger: 05-03 23:14:44] {3097} INFO -  at 1.4s,	estimator histgb's best error=7.9760e-01,	best estimator lgbm's best error=7.9332e-01
[flaml.automl.logger: 05-03 23:14:44] {2911} INFO - iteration 3, current learner svc
[flaml.automl.logger: 05-03 23:14:44] {3097} INFO -  at 1.5s,	estimator svc's best error=7.1212e-01,	best estimator svc's best error=7.1212e-01
[flaml.automl.logger: 05-03 23:14:44] {2911} INFO - iteration 4, current learner svc
[flaml.automl.logger: 05-03 23:14:46] {3097} INFO -  at 3.2s,	estimator svc's best error=7.1212e-01,	best estimator svc's best error=7.1212e-01
[flaml.automl.logger: 05-03 23:14:46] {2911} INFO - iteration 5, current learner sgd
[flaml.automl.logger: 05-03 23:14:47] {3097} INFO -  at 4.0s,	estimator sgd's best error=7.9716e-01,	best estimator svc's best error=7.1212e-01
[flaml.automl.logger: 05-03 23:14:47] {2911} INFO - iteration 6, current learner lgbm
[flaml.automl.logger: 05-03 23:14:47] {3097} INFO -  at 4.4s,	estimator lgbm's 